# Interactive GT Mask Correction and Progress Tracker

This notebook provides an interactive display to align the Summer GT mask over Summer or Winter RGB images, save corrected RGB/depth/GT files, and track which trees remain uncorrected.

## Output folders

```text
Image Correction/
├── Summer RGB/
├── Summer D/
├── Summer GT/
├── Winter RGB/
├── Winter D/
└── Winter GT/
```

## Features
- RGB is centre-cropped to **480 × 480**, then resized to **256 × 256**.
- Matching depth image uses the same crop coordinates, then resizes to 256 × 256.
- Interactive sliders move the GT overlay horizontally/vertically and adjust opacity.
- Buttons save the current corrected set or reset the shift.
- Summer and Winter can be selected separately.
- A live progress panel lists completed and remaining tree IDs for the selected season.
- The image dropdown hides trees that already have a complete saved RGB + depth + GT set; after saving, it advances to the next unfinished tree.
- Saving the same tree and season again overwrites the existing files instead of making duplicates.

**Progress definition:** a tree is marked corrected for a season when all three corresponding output files (RGB, D, GT) exist in that season's output folders. The progress panel refreshes when you save and when you switch seasons. Save this notebook in `Image Correction/Notebooks/`. The code uses the main project folder for datasets and saves corrected files into the sibling `Image Correction` folders, not inside `Notebooks`. Run all cells in order. If widgets do not render, use JupyterLab/Notebook with `ipywidgets` enabled.

In [8]:
# If ipywidgets is not installed, uncomment and run:
# %pip install ipywidgets

from pathlib import Path
import re
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

%matplotlib inline


In [9]:
# ============================================================
# 1. PATHS AND SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3"
    r"\Numerical Methods and Machine Learning\Major Project"
)
SORTED_ROOT = PROJECT_ROOT / "Dataset (With Summer GT) - Sorted"
UNSORTED_ROOT = PROJECT_ROOT / "Dataset (With Summer GT)"
DATASET_ROOT = SORTED_ROOT if SORTED_ROOT.exists() else UNSORTED_ROOT

SUMMER_RGB_DIR = DATASET_ROOT / "Summer RGB"
SUMMER_D_DIR = DATASET_ROOT / "Summer D"
WINTER_RGB_DIR = DATASET_ROOT / "Winter RGB"
WINTER_D_DIR = DATASET_ROOT / "Winter D"
GT_DIR = DATASET_ROOT / "Summer GT"

OUTPUT_ROOT = PROJECT_ROOT / "Image Correction"
OUTPUT_DIRS = {
    ("Summer", "RGB"): OUTPUT_ROOT / "Summer RGB",
    ("Summer", "D"):   OUTPUT_ROOT / "Summer D",
    ("Summer", "GT"):  OUTPUT_ROOT / "Summer GT",
    ("Winter", "RGB"): OUTPUT_ROOT / "Winter RGB",
    ("Winter", "D"):   OUTPUT_ROOT / "Winter D",
    ("Winter", "GT"):  OUTPUT_ROOT / "Winter GT",
}
for folder in OUTPUT_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

CROP_SIZE = 480
DISPLAY_SIZE = 256
CROP_CENTER_X = None  # None = image centre; otherwise original RGB x-coordinate
CROP_CENTER_Y = None  # None = image centre; otherwise original RGB y-coordinate

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

print("Dataset root:", DATASET_ROOT)
for label, folder in [
    ("Summer RGB", SUMMER_RGB_DIR), ("Summer D", SUMMER_D_DIR),
    ("Summer GT", GT_DIR), ("Winter RGB", WINTER_RGB_DIR),
    ("Winter D", WINTER_D_DIR)
]:
    print(f"{label}: {'found' if folder.exists() else 'MISSING'} — {folder}")
print("Output root:", OUTPUT_ROOT)


Dataset root: C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\Dataset (With Summer GT) - Sorted
Summer RGB: found — C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\Dataset (With Summer GT) - Sorted\Summer RGB
Summer D: found — C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\Dataset (With Summer GT) - Sorted\Summer D
Summer GT: found — C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\Dataset (With Summer GT) - Sorted\Summer GT
Winter RGB: found — C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Major Project\Dataset (With Summer GT) - Sorted\Winter RGB
Winter D: found — C:\Users\riley\OneDrive\Monash Bachelor of Engineering\Year 3\Numerical Methods and Machine Learning\Majo

In [10]:
# ============================================================
# 2. MATCH RGB + DEPTH + GT BY TREE ID
# ============================================================

def get_tree_id(filename):
    match = re.search(r"(R\d+N\d+)", filename.upper())
    return match.group(1) if match else None

def list_images(folder):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

def index_by_tree_id(folder):
    candidates = {}
    for path in list_images(folder):
        tree_id = get_tree_id(path.name)
        if tree_id:
            candidates.setdefault(tree_id, []).append(path)
    unique = {k: v[0] for k, v in candidates.items() if len(v) == 1}
    ambiguous = sorted(k for k, v in candidates.items() if len(v) > 1)
    return unique, ambiguous

gt_by_id, ambiguous_gt = index_by_tree_id(GT_DIR)
summer_d_by_id, ambiguous_summer_d = index_by_tree_id(SUMMER_D_DIR)
winter_d_by_id, ambiguous_winter_d = index_by_tree_id(WINTER_D_DIR)

def build_pairs(rgb_folder, depth_by_id, season):
    pairs = []
    for rgb_path in list_images(rgb_folder):
        tree_id = get_tree_id(rgb_path.name)
        if tree_id and tree_id in gt_by_id and tree_id in depth_by_id:
            pairs.append({
                "season": season,
                "tree_id": tree_id,
                "rgb_path": rgb_path,
                "depth_path": depth_by_id[tree_id],
                "gt_path": gt_by_id[tree_id],
                "label": f"{tree_id} | {rgb_path.name}"
            })
    return pairs

PAIRS = (
    build_pairs(SUMMER_RGB_DIR, summer_d_by_id, "Summer")
    + build_pairs(WINTER_RGB_DIR, winter_d_by_id, "Winter")
)

if not PAIRS:
    raise RuntimeError(
        "No complete RGB/depth/GT matches found. Check the dataset folder paths "
        "and that filenames contain the same IDs, e.g. R01N01."
    )

print(f"Matched image sets: {len(PAIRS)}")
if ambiguous_gt:
    print("Ambiguous GT IDs excluded:", ambiguous_gt[:20])
if ambiguous_summer_d:
    print("Ambiguous Summer D IDs excluded:", ambiguous_summer_d[:20])
if ambiguous_winter_d:
    print("Ambiguous Winter D IDs excluded:", ambiguous_winter_d[:20])


Matched image sets: 1124


In [11]:
# ============================================================
# 3. IMAGE PREPARATION: 480x480 CROP THEN 256x256 RESIZE
# ============================================================

def crop_square(image, crop_size=CROP_SIZE, center_x=None, center_y=None):
    h, w = image.shape[:2]
    if h < crop_size or w < crop_size:
        raise ValueError(f"Image {w}x{h} is smaller than crop {crop_size}x{crop_size}.")
    cx = w // 2 if center_x is None else int(center_x)
    cy = h // 2 if center_y is None else int(center_y)
    x0 = max(0, min(cx - crop_size // 2, w - crop_size))
    y0 = max(0, min(cy - crop_size // 2, h - crop_size))
    return image[y0:y0 + crop_size, x0:x0 + crop_size].copy(), (x0, y0)

def load_pair(pair):
    rgb_bgr = cv2.imread(str(pair["rgb_path"]), cv2.IMREAD_COLOR)
    depth = cv2.imread(str(pair["depth_path"]), cv2.IMREAD_UNCHANGED)
    gt_gray = cv2.imread(str(pair["gt_path"]), cv2.IMREAD_GRAYSCALE)
    if rgb_bgr is None:
        raise ValueError(f"Could not read RGB: {pair['rgb_path']}")
    if depth is None:
        raise ValueError(f"Could not read depth: {pair['depth_path']}")
    if gt_gray is None:
        raise ValueError(f"Could not read GT: {pair['gt_path']}")

    rgb_crop, (x0, y0) = crop_square(
        rgb_bgr, CROP_SIZE, CROP_CENTER_X, CROP_CENTER_Y
    )

    if depth.shape[:2] == rgb_bgr.shape[:2]:
        depth_crop = depth[y0:y0 + CROP_SIZE, x0:x0 + CROP_SIZE].copy()
    else:
        rh, rw = rgb_bgr.shape[:2]
        dh, dw = depth.shape[:2]
        dx0, dy0 = round(x0 * dw / rw), round(y0 * dh / rh)
        dcw, dch = round(CROP_SIZE * dw / rw), round(CROP_SIZE * dh / rh)
        depth_crop = depth[dy0:dy0 + dch, dx0:dx0 + dcw].copy()
        if depth_crop.size == 0:
            raise ValueError(f"Depth crop empty: {pair['depth_path']}")

    rgb_small = cv2.resize(rgb_crop, (DISPLAY_SIZE, DISPLAY_SIZE),
                            interpolation=cv2.INTER_LINEAR)
    depth_small = cv2.resize(depth_crop, (DISPLAY_SIZE, DISPLAY_SIZE),
                              interpolation=cv2.INTER_NEAREST)
    gt_small = cv2.resize(gt_gray, (DISPLAY_SIZE, DISPLAY_SIZE),
                          interpolation=cv2.INTER_NEAREST)

    rgb = cv2.cvtColor(rgb_small, cv2.COLOR_BGR2RGB)
    # Assumes bright pixels in the GT indicate foreground.
    gt = (gt_small > 127).astype(np.uint8)
    return rgb, depth_small, gt, (x0, y0)

def shift_mask(mask, dx, dy):
    matrix = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(mask, matrix, (mask.shape[1], mask.shape[0]),
                          flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def make_overlay(rgb, mask, alpha=0.45):
    overlay = rgb.astype(np.float32).copy()
    foreground = mask > 0
    magenta = np.array([255, 0, 255], dtype=np.float32)
    overlay[foreground] = (1 - alpha) * overlay[foreground] + alpha * magenta
    return np.clip(overlay, 0, 255).astype(np.uint8)

def depth_for_display(depth):
    if depth.ndim == 2:
        d = depth.astype(np.float32)
        finite = np.isfinite(d)
        if not finite.any():
            return np.zeros((*d.shape, 3), dtype=np.uint8)
        lo, hi = np.percentile(d[finite], [1, 99])
        if hi <= lo:
            hi = lo + 1
        scaled = np.clip((d - lo) / (hi - lo), 0, 1)
        scaled[~finite] = 0
        return cv2.cvtColor((scaled * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if depth.ndim == 3 and depth.shape[2] == 3:
        return cv2.cvtColor(depth, cv2.COLOR_BGR2RGB)
    return np.zeros((DISPLAY_SIZE, DISPLAY_SIZE, 3), dtype=np.uint8)


In [12]:
# ============================================================
# 4. INTERACTIVE WIDGETS
# ============================================================

season_widget = widgets.ToggleButtons(
    options=["Summer", "Winter"], value="Summer", description="Season:"
)
image_widget = widgets.Dropdown(
    options=[], description="Image:", layout=widgets.Layout(width="95%")
)
dx_widget = widgets.IntSlider(
    value=0, min=-128, max=128, step=1, description="GT shift X:",
    continuous_update=True, style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)
dy_widget = widgets.IntSlider(
    value=0, min=-128, max=128, step=1, description="GT shift Y:",
    continuous_update=True, style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)
alpha_widget = widgets.FloatSlider(
    value=0.45, min=0, max=1, step=0.05, description="GT opacity:",
    continuous_update=True, readout_format=".2f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)
reset_button = widgets.Button(description="Reset shift", button_style="warning")
save_button = widgets.Button(description="Save corrected RGB + D + GT",
                             button_style="success")
refresh_button = widgets.Button(description="Refresh progress")
status_output = widgets.Output()
plot_output = widgets.Output()
progress_output = widgets.Output()
shift_records = {}


In [13]:
# ============================================================
# 5. PROGRESS TRACKER + FILTER COMPLETED IMAGES FROM DROPDOWN
# ============================================================

def output_paths(pair):
    # Stable filenames prevent duplicate copies: saving the same tree+season overwrites.
    tree_id, season = pair["tree_id"], pair["season"]
    return {
        "RGB": OUTPUT_DIRS[(season, "RGB")] / f"{tree_id}_{season}_RGB.png",
        "D":   OUTPUT_DIRS[(season, "D")] / f"{tree_id}_{season}_D.png",
        "GT":  OUTPUT_DIRS[(season, "GT")] / f"{tree_id}_{season}_GT.png",
    }

def is_pair_corrected(pair):
    # Consider a tree complete only when all 3 output files exist.
    return all(p.exists() for p in output_paths(pair).values())

def current_pair():
    return next((p for p in PAIRS if p["label"] == image_widget.value), None)

def shift_key(pair):
    return (pair["season"], pair["tree_id"])

def refresh_progress(*args):
    season = season_widget.value
    season_pairs = [p for p in PAIRS if p["season"] == season]

    # Use one entry per tree ID in progress reporting.
    by_tree = {}
    for pair in season_pairs:
        by_tree.setdefault(pair["tree_id"], pair)

    corrected = sorted(tree_id for tree_id, pair in by_tree.items()
                       if is_pair_corrected(pair))
    remaining = sorted(set(by_tree) - set(corrected))

    with progress_output:
        clear_output(wait=True)
        print(f"{season} correction progress")
        print(f"Completed: {len(corrected)} / {len(by_tree)} trees")
        if by_tree:
            print(f"Progress: {100 * len(corrected) / len(by_tree):.1f}%")
        print("\\nTrees still needing correction:")
        print(", ".join(remaining) if remaining else "None — all matched trees are corrected!")
        print("\\nCompleted trees:")
        print(", ".join(corrected) if corrected else "None yet")

def update_image_options(*args):
    season = season_widget.value
    season_pairs = [p for p in PAIRS if p["season"] == season]

    # Hide pairs already fully saved in the output folders.
    incomplete_pairs = [p for p in season_pairs if not is_pair_corrected(p)]

    # If the current selection becomes complete, switch to the next incomplete pair.
    current_label = image_widget.value
    options = [(p["label"], p["label"]) for p in incomplete_pairs]
    image_widget.options = options

    if options:
        option_values = [value for _, value in options]
        image_widget.value = current_label if current_label in option_values else options[0][1]
    else:
        image_widget.value = None

    refresh_progress()

def remember_shift(*args):
    pair = current_pair()
    if pair is not None:
        shift_records[shift_key(pair)] = (dx_widget.value, dy_widget.value)

def restore_current_shift(*args):
    pair = current_pair()
    if pair is not None:
        dx, dy = shift_records.get(shift_key(pair), (0, 0))
        dx_widget.value = dx
        dy_widget.value = dy


In [14]:
# ============================================================
# 6. DISPLAY / SAVE HANDLERS
# ============================================================

def draw(*args):
    pair = current_pair()
    with plot_output:
        clear_output(wait=True)
        if pair is None:
            print("No image selected.")
            return
        try:
            rgb, depth, gt, crop_origin = load_pair(pair)
        except Exception as exc:
            print("Could not load this image pair:", exc)
            return

        shifted_gt = shift_mask(gt, dx_widget.value, dy_widget.value)
        overlay = make_overlay(rgb, shifted_gt, alpha_widget.value)
        depth_rgb = depth_for_display(depth)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4.7))
        axes[0].imshow(rgb)
        axes[0].set_title("RGB (480 crop → 256)")
        axes[1].imshow(depth_rgb)
        axes[1].set_title("Matching depth image")
        axes[2].imshow(overlay)
        axes[2].set_title(f"GT overlay: dx={dx_widget.value}, dy={dy_widget.value}")
        axes[3].imshow(shifted_gt, cmap="gray", vmin=0, vmax=1)
        axes[3].set_title("Shifted GT mask")
        for ax in axes:
            ax.axis("off")
        fig.suptitle(
            f"{pair['season']} | {pair['tree_id']} | crop origin x={crop_origin[0]}, y={crop_origin[1]}\\n"
            f"RGB: {pair['rgb_path'].name} | GT source: {pair['gt_path'].name}",
            fontsize=9
        )
        plt.tight_layout()
        plt.show()

def on_reset(button):
    dx_widget.value = 0
    dy_widget.value = 0
    remember_shift()

def on_save(button):
    pair = current_pair()
    if pair is None:
        return
    try:
        rgb, depth, gt, crop_origin = load_pair(pair)
        shifted_gt = shift_mask(gt, dx_widget.value, dy_widget.value)
        rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        paths = output_paths(pair)

        # Lossless PNG output. Existing same tree+season files are overwritten.
        if not cv2.imwrite(str(paths["RGB"]), rgb_bgr):
            raise IOError(f"Failed to save RGB: {paths['RGB']}")
        if not cv2.imwrite(str(paths["D"]), depth):
            raise IOError(f"Failed to save depth: {paths['D']}")
        if not cv2.imwrite(str(paths["GT"]), (shifted_gt * 255).astype(np.uint8)):
            raise IOError(f"Failed to save GT: {paths['GT']}")

        remember_shift()
        with status_output:
            clear_output(wait=True)
            print(f"Saved corrected {pair['season']} set for {pair['tree_id']}:")
            print("RGB:", paths["RGB"])
            print("D:  ", paths["D"])
            print("GT: ", paths["GT"])
            print(f"Shift: dx={dx_widget.value}, dy={dy_widget.value}")
            print("Saving this same tree/season again overwrites these files.")

        refresh_progress()
        # Remove the completed tree from the dropdown and select the next incomplete one.
        update_image_options()
    except Exception as exc:
        with status_output:
            clear_output(wait=True)
            print("Save failed:", exc)

season_widget.observe(update_image_options, names="value")
image_widget.observe(restore_current_shift, names="value")
for control in [dx_widget, dy_widget, alpha_widget]:
    control.observe(remember_shift, names="value")
    control.observe(draw, names="value")
season_widget.observe(draw, names="value")
image_widget.observe(draw, names="value")
reset_button.on_click(on_reset)
save_button.on_click(on_save)
refresh_button.on_click(refresh_progress)

update_image_options()
restore_current_shift()
draw()
refresh_progress()

display(widgets.VBox([
    season_widget,
    image_widget,
    dx_widget,
    dy_widget,
    alpha_widget,
    widgets.HBox([reset_button, save_button, refresh_button]),
    widgets.HTML("<h3>Correction progress</h3>"),
    progress_output,
    status_output,
    plot_output
]))


## Notes and troubleshooting

- If the controls show but the images do not update, ensure you are running the notebook in JupyterLab/Notebook with `ipywidgets` enabled.
- Positive X moves the GT right; positive Y moves it down.
- RGB and depth are cropped together, using the same coordinates when dimensions match.
- The GT mask is resized to 256 × 256 and translated. It is not cropped to the same 480 × 480 window, because the GT is assumed to already represent the corresponding tree view.
- The progress list counts a tree as complete only if its RGB, depth, and GT output files all exist.
- The image dropdown only lists unfinished tree/season pairs. After saving a complete set, the saved tree disappears from the dropdown and the notebook selects the next unfinished pair. Switch Summer/Winter to work through each season separately.
- Progress is tracked separately for Summer and Winter. The same tree may need correction once for each season.
- Files are saved as PNG with stable names such as `R01N01_Summer_RGB.png`, `R01N01_Summer_D.png`, and `R01N01_Summer_GT.png`. Saving again overwrites those exact files.
- If the GT uses black for foreground and white for background, invert the thresholding in `load_pair()`.
- If your filenames do not share tree IDs like `R01N01`, adjust `get_tree_id()` and the matching logic.